In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

files_info = [
    ("metric.csv", "kpi_name"),
    ("trace.csv", "trace_name"),
    ("log.csv", "log_name"),
    ("error_logs.csv", "message"),
]

summary_rows = []

for fname, name_col in files_info:
    # load
    df = pd.read_csv(fname)
    row_count = len(df)
    # timestamps: handle missing column gracefully
    if "timestamp" in df.columns and row_count > 0:
        ts_series = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.tz_convert(tz)
        min_ts = ts_series.min()
        max_ts = ts_series.max()
        # convert to compact ISO strings
        min_ts_s = None if pd.isna(min_ts) else min_ts.isoformat()
        max_ts_s = None if pd.isna(max_ts) else max_ts.isoformat()
    else:
        min_ts_s = None
        max_ts_s = None

    # top cmdb_id values by frequency (up to 20)
    if "cmdb_id" in df.columns and row_count > 0:
        top_cmdb = df["cmdb_id"].dropna().astype(str).value_counts().head(20).index.tolist()
    else:
        top_cmdb = []

    # top name values (kpi/trace/log/message) by frequency (up to 20)
    if name_col in df.columns and row_count > 0:
        top_names = df[name_col].dropna().astype(str).value_counts().head(20).index.tolist()
    else:
        top_names = []

    summary_rows.append({
        "filename": fname,
        "row_count": row_count,
        "min_timestamp(Asia/Shanghai)": min_ts_s,
        "max_timestamp(Asia/Shanghai)": max_ts_s,
        "top_cmdb_id_up_to_20": top_cmdb,
        "top_names_up_to_20": top_names
    })

summary_df = pd.DataFrame(summary_rows)[
    ["filename", "row_count", "min_timestamp(Asia/Shanghai)", "max_timestamp(Asia/Shanghai)",
     "top_cmdb_id_up_to_20", "top_names_up_to_20"]
]

summary_df

```
Out[1]:
```
```python
# Compose a concise plain-English summary based on the previously computed summary_df.
# If summary_df is not present in the kernel, fall back to reloading the CSVs and recomputing a compact summary.

import pandas as pd

def compact_list_repr(lst, max_items=6):
    if not lst:
        return "[]"
    if len(lst) <= max_items:
        return ", ".join(lst)
    return ", ".join(lst[:max_items]) + ", ..."

try:
    summary_df  # reuse existing variable
except NameError:
    # fallback: recreate a minimal summary if summary_df isn't available
    files_info = [
        ("metric.csv", "kpi_name"),
        ("trace.csv", "trace_name"),
        ("log.csv", "log_name"),
        ("error_logs.csv", "message"),
    ]
    rows = []
    for fname, name_col in files_info:
        df = pd.read_csv(fname)
        row_count = len(df)
        if row_count:
            ts_series = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.tz_convert("Asia/Shanghai")
            min_ts = ts_series.min().isoformat()
            max_ts = ts_series.max().isoformat()
        else:
            min_ts = max_ts = None
        top_cmdb = df["cmdb_id"].dropna().astype(str).value_counts().head(20).index.tolist() if "cmdb_id" in df.columns else []
        top_names = df[name_col].dropna().astype(str).value_counts().head(20).index.tolist() if name_col in df.columns else []
        rows.append({
            "filename": fname,
            "row_count": row_count,
            "min_ts": min_ts,
            "max_ts": max_ts,
            "top_cmdb": top_cmdb,
            "top_names": top_names
        })
    summary_df = pd.DataFrame(rows)

lines = []
for _, r in summary_df.iterrows():
    fname = r["filename"]
    row_count = r["row_count"]
    min_ts = r["min_ts"]
    max_ts = r["max_ts"]
    top_cmdb = r["top_cmdb"]
    top_names = r["top_names"]
    if fname == "error_logs.csv":
        names_label = "top messages"
    elif fname == "metric.csv":
        names_label = "top KPI names"
    elif fname == "trace.csv":
        names_label = "top trace names"
    else:
        names_label = "top log names"
    lines.append(
        f"{fname}: {row_count} rows; timestamps {min_ts} to {max_ts}; "
        f"top cmdb_id(s): {compact_list_repr(top_cmdb)}; {names_label}: {compact_list_repr(top_names)}"
    )

summary_text = "\n".join(lines)

# Display the concise plain-English summary
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

filename  row_count min_timestamp(Asia/Shanghai) max_timestamp(Asia/Shanghai)                               top_cmdb_id_up_to_20                                 top_names_up_to_20
0      metric.csv     321716    2022-03-21T04:00:00+08:00    2022-03-21T05:00:00+08:00  [adservice, adservice2, frontend2-0, frontend-...  [container.node-6.container_network_transmit_e...
1       trace.csv      28732    2022-03-21T04:30:00+08:00    2022-03-21T04:59:00+08:00  [frontend-0, frontend-1, frontend-2, checkouts...  [trace.from_frontend-0.duration_mean, trace.fr...
2         log.csv       1732    2022-03-21T04:30:00+08:00    2022-03-21T05:00:00+08:00  [cartservice-0, cartservice-2, currencyservice...                   [log.error_count, log.row_count]
3  error_logs.csv          1    2022-03-21T04:57:13+08:00    2022-03-21T04:57:13+08:00                                       [frontend-2]          [severity: error, message: request error]```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load metric.csv (reuse metric_df if present)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is datetime with Asia/Shanghai tz
if metric_df['timestamp'].dtype.kind in {'i', 'u', 'f'}:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # fallback if already string
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], utc=True).dt.tz_convert(tz)

# Ensure value is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global stats per (cmdb_id, kpi_name) using the full series
group_cols = ['cmdb_id', 'kpi_name']
global_stats = metric_df.groupby(group_cols)['value'].agg(
    count_global='count',
    global_P95=lambda s: s.quantile(0.95),
    global_P5=lambda s: s.quantile(0.05),
    global_max='max',
    global_min='min'
).reset_index()

# 2) Filter metric rows to the incident window (inclusive) in Asia/Shanghai
start = tz.localize(pd.to_datetime("2022-03-21 04:30:00"))
end = tz.localize(pd.to_datetime("2022-03-21 05:00:00"))
window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()

# 3) For each pair compute window-level aggregations and anomaly counts using global thresholds
# Merge global thresholds into window records to evaluate anomalies
window_with_thresholds = window_df.merge(
    global_stats,
    on=group_cols,
    how='left',
    suffixes=('', '_glob')
)

# Boolean anomaly flags
window_with_thresholds['is_above_p95'] = window_with_thresholds['value'] > window_with_thresholds['global_P95']
window_with_thresholds['is_below_p5'] = window_with_thresholds['value'] < window_with_thresholds['global_P5']
window_with_thresholds['is_anom'] = window_with_thresholds['is_above_p95'] | window_with_thresholds['is_below_p5']

# Aggregate per pair within window
agg_window = window_with_thresholds.groupby(group_cols).agg(
    count_in_window=('value', 'count'),
    count_above_p95=('is_above_p95', 'sum'),
    count_below_p5=('is_below_p5', 'sum'),
    earliest_anom_ts=('ts', lambda s: s[window_with_thresholds.loc[s.index, 'is_anom']].min() if len(s)>0 else pd.NaT),
    max_in_window=('value', 'max'),
    min_in_window=('value', 'min')
).reset_index()

# For pairs that have zero rows in the window, ensure they appear with zeros/NaNs
pair_table_full = global_stats.merge(agg_window, on=group_cols, how='left')

# Fill NaNs for counts with 0
pair_table_full['count_in_window'] = pair_table_full['count_in_window'].fillna(0).astype(int)
pair_table_full['count_above_p95'] = pair_table_full['count_above_p95'].fillna(0).astype(int)
pair_table_full['count_below_p5'] = pair_table_full['count_below_p5'].fillna(0).astype(int)

# Convert earliest_anom_ts to ISO strings in Asia/Shanghai or None
def ts_to_iso(x):
    if pd.isna(x):
        return None
    if x.tzinfo is None:
        x = tz.localize(x)
    return x.isoformat()

pair_table_full['earliest_anom_ts'] = pair_table_full['earliest_anom_ts'].apply(ts_to_iso)

# Keep numeric window max/min as-is (may be NaN if no data in window)
# Compute sort key: max of anomaly counts
pair_table_full['anom_count_max'] = pair_table_full[['count_above_p95', 'count_below_p5']].max(axis=1)

# Prepare compact result table with requested columns
result_cols = [
    'cmdb_id', 'kpi_name',
    'global_P95', 'global_P5',
    'count_in_window', 'count_above_p95', 'count_below_p5',
    'earliest_anom_ts', 'max_in_window', 'min_in_window'
]
result_df = pair_table_full[result_cols].copy()

# Rank by anomaly count descending, show pairs with any anomalies first
result_df = result_df.sort_values(by=['count_above_p95', 'count_below_p5', 'count_in_window'], ascending=[False, False, False])

# Show up to top 20 pairs
top20_pairs = result_df.head(20)

# Expose the full intermediate per-pair table for follow-up as required (but do not print it)
# pair_table_full is already stored.

top20_pairs

```
Out[2]:
```
```python
# Compose a concise plain-English summary of the previously displayed top-20 anomaly table.
import pandas as pd

try:
    top20_pairs
except NameError:
    raise RuntimeError("Expected variable top20_pairs not found in kernel.")

# Prepare a short list of example pairs (up to 10) from the top results
examples = []
for _, r in top20_pairs.head(10).iterrows():
    examples.append(f"{r['cmdb_id']} :: {r['kpi_name']}")

# Determine common anomaly counts and window stats among the shown rows
count_in_window_vals = top20_pairs['count_in_window'].unique()
count_above_vals = top20_pairs['count_above_p95'].unique()
count_below_vals = top20_pairs['count_below_p5'].unique()

# earliest anomaly timestamps range among the displayed rows (ignore None)
ts_series = pd.to_datetime(top20_pairs['earliest_anom_ts'].dropna())
earliest_ts_min = ts_series.min().isoformat() if not ts_series.empty else "N/A"
earliest_ts_max = ts_series.max().isoformat() if not ts_series.empty else "N/A"

summary = (
    "Summary of metric anomaly analysis for 2022-03-21 04:30:00+08 through 2022-03-21 05:00:00+08:\n"
    "- Top pairs shown (up to 20) were ranked by the maximum of count_above_p95 and count_below_p5.\n"
    f"- For the displayed top pairs, count_in_window values = {list(count_in_window_vals)} (most shown = 31).\n"
    f"- count_above_p95 values = {list(count_above_vals)}, count_below_p5 values = {list(count_below_vals)} "
    "(these top rows each show 3 samples above P95 and 3 below P5).\n"
    f"- Earliest anomaly timestamps among the displayed rows range from {earliest_ts_min} to {earliest_ts_max}.\n"
    "- Example top anomalous (cmdb_id :: kpi_name) pairs:\n  " + "\n  ".join(examples) + "\n"
    "- Interpretation: multiple (cmdb_id, kpi) series experienced both high and low excursions "
    "(3 samples above their global P95 and 3 below their global P5) within the incident window. "
    "These pairs should be prioritized for investigation."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name     global_P95      global_P5  count_in_window  count_above_p95  count_below_p5           earliest_anom_ts  max_in_window  min_in_window
922           adservice2     runtime.jvm_memory_pool_MB_used.Survivor_Space       0.222586       0.156352               31                3               3  2022-03-21T04:43:00+08:00       0.241723       0.155144
990         adservice2-0  container.node-6.container_network_receive_pac...     563.500000     435.000000               31                3               3  2022-03-21T04:33:00+08:00     621.500000     397.000000
992         adservice2-0  container.node-6.container_network_transmit_MB...       1.319525       1.260567               31                3               3  2022-03-21T04:32:00+08:00       1.588862       0.970401
1448      cartservice2-0  mesh.source.cartservice2.redis-cart2.istio_tcp...    3451.500000    2336.500000               31                3               3  2022-03-21T04:31:00+08:00    3820.500000    2196.500000
1449      cartservice2-0  mesh.source.cartservice2.redis-cart2.istio_tcp...    1756.500000     944.000000               31                3               3  2022-03-21T04:31:00+08:00    1827.000000     895.000000
1883  checkoutservice2-0       container.node-6.container_cpu_usage_seconds       0.080673       0.014241               31                3               3  2022-03-21T04:46:00+08:00       0.094460       0.000093
1971  checkoutservice2-0  mesh.source.checkoutservice2.currencyservice2....       0.008350       0.001575               31                3               3  2022-03-21T04:46:00+08:00       0.011450       0.000000
1991  checkoutservice2-0  mesh.source.checkoutservice2.paymentservice2.i...       0.003625       0.000525               31                3               3  2022-03-21T04:46:00+08:00       0.005150       0.000000
2318  currencyservice2-0  container.node-6.container_network_transmit_MB...       0.872553       0.781968               31                3               3  2022-03-21T04:40:00+08:00       0.880629       0.753258
2339  currencyservice2-0  mesh.destination.checkoutservice2.currencyserv...       0.004725       0.001050               31                3               3  2022-03-21T04:41:00+08:00       0.007875       0.000000
2345  currencyservice2-0  mesh.destination.frontend2.currencyservice2.is...   84702.500000   57285.000000               31                3               3  2022-03-21T04:35:00+08:00   93752.500000   45422.500000
2350  currencyservice2-0  mesh.destination.frontend2.currencyservice2.is...     138.500000      92.000000               31                3               3  2022-03-21T04:35:00+08:00     149.000000      65.500000
2352  currencyservice2-0  mesh.destination.frontend2.currencyservice2.is...     138.500000      92.000000               31                3               3  2022-03-21T04:35:00+08:00     149.000000      65.500000
2355  currencyservice2-0  mesh.destination.frontend2.currencyservice2.is...  180725.000000  123000.000000               31                3               3  2022-03-21T04:35:00+08:00  199725.000000   98025.000000
2357  currencyservice2-0  mesh.destination.frontend2.currencyservice2.is...     138.500000      92.000000               31                3               3  2022-03-21T04:35:00+08:00     149.000000      65.500000
2520      emailservice-2        container.node-6.container_cpu_user_seconds       0.020000       0.010000               31                3               3  2022-03-21T04:31:00+08:00       0.025000       0.005000
2804          frontend-0  mesh.source.frontend.currencyservice.istio_res...   34237.500000   23925.000000               31                3               3  2022-03-21T04:38:00+08:00   39675.000000   12975.000000
3208          frontend-2  mesh.source.frontend.recommendationservice.ist...       0.062225       0.029350               31                3               3  2022-03-21T04:37:00+08:00      30.517200       0.004050
3335         frontend2-0  mesh.source.frontend2.adservice2.istio_respons...    9510.000000    6062.500000               31                3               3  2022-03-21T04:35:00+08:00   10740.000000    4435.000000
3373         frontend2-0  mesh.source.frontend2.currencyservice2.istio_r...  214675.000000  142600.000000               31                3               3  2022-03-21T04:35:00+08:00  230950.000000  101525.000000```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing per-pair table if present under common names; otherwise recompute
per_pair_candidates = ['pair_table_full', 'per_pair_table', 'per_pair_df', 'pair_table']
for name in per_pair_candidates:
    if name in globals():
        per_pair_df = globals()[name]
        break
else:
    # Recompute per-pair table (uses metric_df if present, otherwise loads metric.csv)
    try:
        metric_df
    except NameError:
        metric_df = pd.read_csv("metric.csv")
    # ensure timestamp and value, timezone
    if metric_df['timestamp'].dtype.kind in {'i','u','f'}:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], utc=True).dt.tz_convert(tz)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    group_cols = ['cmdb_id', 'kpi_name']
    global_stats = metric_df.groupby(group_cols)['value'].agg(
        count_global='count',
        global_P95=lambda s: s.quantile(0.95),
        global_P5=lambda s: s.quantile(0.05),
        global_max='max',
        global_min='min'
    ).reset_index()
    start = tz.localize(pd.to_datetime("2022-03-21 04:30:00"))
    end = tz.localize(pd.to_datetime("2022-03-21 05:00:00"))
    window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()
    window_with_thresholds = window_df.merge(global_stats, on=group_cols, how='left')
    window_with_thresholds['is_above_p95'] = window_with_thresholds['value'] > window_with_thresholds['global_P95']
    window_with_thresholds['is_below_p5'] = window_with_thresholds['value'] < window_with_thresholds['global_P5']
    window_with_thresholds['is_anom'] = window_with_thresholds['is_above_p95'] | window_with_thresholds['is_below_p5']
    agg_window = window_with_thresholds.groupby(group_cols).agg(
        count_in_window=('value', 'count'),
        count_above_p95=('is_above_p95', 'sum'),
        count_below_p5=('is_below_p5', 'sum'),
        earliest_anom_ts=('ts', lambda s: s[window_with_thresholds.loc[s.index, 'is_anom']].min() if len(s)>0 else pd.NaT),
        max_in_window=('value', 'max'),
        min_in_window=('value', 'min')
    ).reset_index()
    per_pair_df = global_stats.merge(agg_window, on=group_cols, how='left')
    per_pair_df['count_in_window'] = per_pair_df['count_in_window'].fillna(0).astype(int)
    per_pair_df['count_above_p95'] = per_pair_df['count_above_p95'].fillna(0).astype(int)
    per_pair_df['count_below_p5'] = per_pair_df['count_below_p5'].fillna(0).astype(int)
    per_pair_df['earliest_anom_ts'] = per_pair_df['earliest_anom_ts'].apply(
        lambda x: x.isoformat() if pd.notna(x) else None
    )
    # store recomputed per_pair_df in kernel under pair_table_full for reuse
    pair_table_full = per_pair_df

# Ensure anom_count and datetime column
per_pair_df = per_pair_df.copy()
per_pair_df['anom_count'] = per_pair_df['count_above_p95'] + per_pair_df['count_below_p5']
# convert earliest_anom_ts to datetime with tz for aggregation
per_pair_df['earliest_anom_dt'] = pd.to_datetime(per_pair_df['earliest_anom_ts'], utc=True).dt.tz_convert(tz)

# Aggregate by cmdb_id
grp = per_pair_df.groupby('cmdb_id', sort=False)

total_anom = grp['anom_count'].sum().rename('total_anom')
num_kpis_with_anom = grp.apply(lambda g: (g['anom_count'] > 0).sum()).rename('num_kpis_with_anom')
earliest_anom_dt = grp['earliest_anom_dt'].min().rename('earliest_anom_dt')
total_count_in_window = grp['count_in_window'].sum().rename('total_count_in_window')

# Identify top KPI per cmdb_id by anom_count (tie-breaker: larger count_in_window)
idx_top = per_pair_df.groupby('cmdb_id').apply(
    lambda g: g.sort_values(['anom_count', 'count_in_window'], ascending=[False, False]).index[0]
).rename('top_idx')

top_kpis = per_pair_df.loc[idx_top.values].set_index(per_pair_df.loc[idx_top.values].index)
top_kpis = top_kpis.reset_index(drop=True)  # align for merging

# Build summary dataframe
cmdb_summary = pd.DataFrame({
    'cmdb_id': total_anom.index,
    'total_anom': total_anom.values,
    'num_kpis_with_anom': num_kpis_with_anom.values,
    'earliest_anom_dt': earliest_anom_dt.values,
    'total_count_in_window': total_count_in_window.values
})

# Merge top_kpi info: need to align by cmdb_id; construct a small df from top_kpis
top_kpi_info = per_pair_df.loc[idx_top.values, ['cmdb_id', 'kpi_name', 'anom_count', 'max_in_window', 'min_in_window']].copy()
top_kpi_info.columns = ['cmdb_id', 'top_kpi_name', 'top_kpi_anom_count', 'top_kpi_max_in_window', 'top_kpi_min_in_window']

cmdb_anom_summary = cmdb_summary.merge(top_kpi_info, on='cmdb_id', how='left')

# Convert earliest_anom_dt to ISO strings in Asia/Shanghai (or None)
def dt_to_iso(x):
    if pd.isna(x):
        return None
    # x is already tz-aware in tz
    return x.isoformat()

cmdb_anom_summary['earliest_anom_ts'] = cmdb_anom_summary['earliest_anom_dt'].apply(dt_to_iso)
cmdb_anom_summary = cmdb_anom_summary[[
    'cmdb_id', 'total_anom', 'num_kpis_with_anom', 'earliest_anom_ts',
    'total_count_in_window', 'top_kpi_name', 'top_kpi_anom_count',
    'top_kpi_max_in_window', 'top_kpi_min_in_window'
]]

# Order by total_anom descending and limit to top 20
cmdb_anom_summary = cmdb_anom_summary.sort_values(by='total_anom', ascending=False).head(20).reset_index(drop=True)

# Store result in kernel variable as requested
# cmdb_anom_summary is now available for follow-up
cmdb_anom_summary

```
Out[3]:
```
```python
# Compose a concise plain-English summary using the cmdb_anom_summary table from the kernel.
try:
    cmdb_anom_summary
except NameError:
    raise RuntimeError("cmdb_anom_summary not found in kernel. Please run the aggregation step first.")

df = cmdb_anom_summary.copy()

# Overall highlights
overall_earliest = df['earliest_anom_ts'].dropna().min()
top5 = df.head(5)

lines = []
lines.append("Aggregated anomaly summary by cmdb_id for the incident window:")
lines.append(f"- Earliest anomaly observed at: {overall_earliest}")
lines.append("- Top affected components (ranked by total_anom):")

for i, r in top5.iterrows():
    lines.append(
        f"  {i+1}. {r['cmdb_id']}: total_anom={int(r['total_anom'])}, "
        f"num_kpis_with_anom={int(r['num_kpis_with_anom'])}, earliest_anom_ts={r['earliest_anom_ts']}; "
        f"top_kpi='{r['top_kpi_name']}' (anom_count={int(r['top_kpi_anom_count'])}, "
        f"max_in_window={r['top_kpi_max_in_window']}, min_in_window={r['top_kpi_min_in_window']})"
    )

lines.append("- Interpretation: These cmdb_ids (notably frontend2-0, frontend-0, checkoutservice2-0, frontend-2, frontend-1) "
             "account for the highest aggregated anomaly counts and should be prioritized for investigation.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_anom  num_kpis_with_anom     earliest_anom_ts  total_count_in_window                                       top_kpi_name  top_kpi_anom_count  top_kpi_max_in_window  top_kpi_min_in_window
0                frontend2-0         250                  66  2022-03-20T20:31:00                   6076  mesh.source.frontend2.adservice2.istio_respons...                   6           10740.000000            4435.000000
1                 frontend-0         229                  64  2022-03-20T20:30:00                   5828  mesh.source.frontend.currencyservice.istio_res...                   6           39675.000000           12975.000000
2         checkoutservice2-0         221                  59  2022-03-20T20:32:00                   4464       container.node-6.container_cpu_usage_seconds                   6               0.094460               0.000093
3                 frontend-2         200                  61  2022-03-20T20:31:00                   5828  mesh.source.frontend.recommendationservice.ist...                   6              30.517200               0.004050
4                 frontend-1         176                  62  2022-03-20T20:30:00                   5828  mesh.destination.unknown.frontend.istio_reques...                   5              67.646750               1.565250
5          checkoutservice-0         167                  57  2022-03-20T20:34:00                   4340       container.node-6.container_cpu_usage_seconds                   5               0.081241               0.000964
6                 adservice2         156                  56  2022-03-20T20:30:00                  10323     runtime.jvm_memory_pool_MB_used.Survivor_Space                   6               0.241723               0.155144
7         currencyservice2-0         129                  31  2022-03-20T20:30:00                   2604  container.node-6.container_network_transmit_MB...                   6               0.880629               0.753258
8   productcatalogservice2-0         115                  34  2022-03-20T20:30:00                   3224  mesh.destination.frontend2.productcatalogservi...                   5          149285.000000           86052.500000
9    productcatalogservice-0         115                  40  2022-03-20T20:30:00                   3720         container.node-6.container_cpu_cfs_periods                   5             197.500000             122.500000
10   productcatalogservice-2         113                  41  2022-03-20T20:30:00                   3596  container.node-6.container_network_receive_MB....                   4               0.978502               0.279804
11   productcatalogservice-1         109                  42  2022-03-20T20:35:00                   3720  container.node-6.container_network_receive_MB....                   5               0.711483               0.285750
12                    node-5         108                  36  2022-03-20T20:30:00                   1748                              system.net.bytes_rcvd                   6           96106.470000           82974.470000
13                    node-2         101                  32  2022-03-20T20:30:00                   1779                                      system.io.w_s                   6              75.500000               0.000000
14            cartservice2-0          94                  30  2022-03-20T20:30:00                   3100  mesh.source.cartservice2.redis-cart2.istio_tcp...                   6            3820.500000            2196.500000
15         currencyservice-2          92                  33  2022-03-20T20:32:00                   2728        container.node-6.container_cpu_user_seconds                   5               0.120000               0.060000
16   recommendationservice-2          90                  27  2022-03-20T20:31:00                   3038  container.node-6.container_network_transmit_pa...                   5             466.000000             182.000000
17                    node-3          89                  31  2022-03-20T20:32:00                   1748                              system.net.bytes_rcvd                   6          205263.960000          105239.460000
18  recommendationservice2-0          85                  25  2022-03-20T20:31:00                   3162  mesh.destination.frontend2.recommendationservi...                   5           24687.500000           13995.000000
19                    node-1          85                  34  2022-03-20T20:30:00                   1779                               system.mem.pct_usage                   6              57.650000              42.320000```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp and ts column with Asia/Shanghai tz
if trace_df['timestamp'].dtype.kind in {'i', 'u', 'f'}:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], utc=True).dt.tz_convert(tz)

# Ensure value numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Get top 10 cmdb_id candidates from cmdb_anom_summary
if 'cmdb_anom_summary' not in globals():
    raise RuntimeError("cmdb_anom_summary not found in kernel. Please run prior aggregation step first.")
top_cmdbs = cmdb_anom_summary['cmdb_id'].head(10).tolist()

# Subset trace data to only those cmdb_ids
trace_subset = trace_df[trace_df['cmdb_id'].isin(top_cmdbs)].copy()

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full available series (before filtering)
group_cols = ['cmdb_id', 'trace_name']
global_stats = trace_subset.groupby(group_cols)['value'].agg(
    global_P95=lambda s: s.quantile(0.95),
    global_P5=lambda s: s.quantile(0.05)
).reset_index()

# 2) Filter trace rows to the incident window (inclusive) in Asia/Shanghai
start = tz.localize(pd.to_datetime("2022-03-21 04:30:00"))
end = tz.localize(pd.to_datetime("2022-03-21 05:00:00"))
trace_window = trace_subset[(trace_subset['ts'] >= start) & (trace_subset['ts'] <= end)].copy()

# 3) Merge thresholds into window rows and compute anomaly flags
window_with_thresholds = trace_window.merge(global_stats, on=group_cols, how='left')
window_with_thresholds['is_above_p95'] = window_with_thresholds['value'] > window_with_thresholds['global_P95']
window_with_thresholds['is_below_p5'] = window_with_thresholds['value'] < window_with_thresholds['global_P5']
window_with_thresholds['is_anom'] = window_with_thresholds['is_above_p95'] | window_with_thresholds['is_below_p5']

# 4) Aggregate per (cmdb_id, trace_name) within the window
agg_window = window_with_thresholds.groupby(group_cols).agg(
    count_in_window=('value', 'count'),
    count_above_p95=('is_above_p95', 'sum'),
    count_below_p5=('is_below_p5', 'sum'),
    earliest_anom_ts=('ts', lambda s: s[window_with_thresholds.loc[s.index, 'is_anom']].min() if len(s)>0 else pd.NaT),
    max_in_window=('value', 'max'),
    min_in_window=('value', 'min')
).reset_index()

# 5) Ensure pairs with no window rows are included (merge global_stats with agg_window)
trace_pair_table = global_stats.merge(agg_window, on=group_cols, how='left')

# Fill missing counts with zeros
trace_pair_table['count_in_window'] = trace_pair_table['count_in_window'].fillna(0).astype(int)
trace_pair_table['count_above_p95'] = trace_pair_table['count_above_p95'].fillna(0).astype(int)
trace_pair_table['count_below_p5'] = trace_pair_table['count_below_p5'].fillna(0).astype(int)

# Convert earliest_anom_ts to ISO string in Asia/Shanghai or None
def to_iso(x):
    if pd.isna(x):
        return None
    if x.tzinfo is None:
        x = tz.localize(x)
    return x.isoformat()

trace_pair_table['earliest_anom_ts'] = trace_pair_table['earliest_anom_ts'].apply(to_iso)

# Prepare final ordering key and select requested columns
trace_pair_table['anom_count_max'] = trace_pair_table[['count_above_p95', 'count_below_p5']].max(axis=1)

result_cols = [
    'cmdb_id', 'trace_name',
    'global_P95', 'global_P5',
    'count_in_window', 'count_above_p95', 'count_below_p5',
    'earliest_anom_ts', 'max_in_window', 'min_in_window'
]
trace_anom_top = trace_pair_table[result_cols].sort_values(
    by=['count_above_p95', 'count_below_p5', 'count_in_window'],
    ascending=[False, False, False]
).head(20).reset_index(drop=True)

# Expose trace_anom_top for follow-up (and display compactly as required)
trace_anom_top

```
Out[4]:
```
```python
# Compose a concise plain-English summary based on trace_anom_top from the kernel.
try:
    trace_anom_top
except NameError:
    raise RuntimeError("trace_anom_top not found in kernel. Please run the trace anomaly analysis step first.")

df = trace_anom_top.copy()

# Examples (up to 8) and general stats
examples = df.head(8).apply(lambda r: f"{r['cmdb_id']} :: {r['trace_name']}", axis=1).tolist()
unique_counts = {
    'rows_shown': df.shape[0],
    'common_count_in_window': sorted(df['count_in_window'].unique()),
    'common_count_above_p95': sorted(df['count_above_p95'].unique()),
    'common_count_below_p5': sorted(df['count_below_p5'].unique())
}
ts_vals = df['earliest_anom_ts'].dropna()
earliest_ts = ts_vals.min() if not ts_vals.empty else "N/A"
latest_ts = ts_vals.max() if not ts_vals.empty else "N/A"

summary = (
    "Trace anomaly summary for top cmdb_id candidates (incident window 2022-03-21 04:30–05:00 +08):\n"
    f"- Rows returned: {unique_counts['rows_shown']}. Most pairs have count_in_window = {unique_counts['common_count_in_window']}.\n"
    f"- For the displayed pairs, anomalies per pair are small but consistent: count_above_p95 = {unique_counts['common_count_above_p95']}, "
    f"count_below_p5 = {unique_counts['common_count_below_p5']} (commonly 2 and 2).\n"
    f"- Earliest anomaly timestamps among these traces range from {earliest_ts} to {latest_ts}.\n"
    "- Examples of affected trace series:\n  " + "\n  ".join(examples) + "\n"
    "- Interpretation: Several trace metrics from frontend and currencyservice-related components show short-duration "
    "excursions both above their global P95 and below their global P5 within the incident window. "
    "Investigate frontend and currencyservice traces and their downstream calls for root cause clues."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  global_P95   global_P5  count_in_window  count_above_p95  count_below_p5           earliest_anom_ts  max_in_window  min_in_window
0   currencyservice2-0             trace.from_frontend2-0.duration_p95    0.000157    0.000138               30                2               2  2022-03-21T04:31:00+08:00       0.000170       0.000133
1   currencyservice2-0                trace.from_frontend2-0.row_count  149.200000   76.900000               30                2               2  2022-03-21T04:31:00+08:00     177.000000      65.000000
2           frontend-0                   trace.from_root.duration_mean    0.369744    0.046128               30                2               2  2022-03-21T04:41:00+08:00       5.700789       0.042793
3           frontend-0                    trace.from_root.duration_p95    2.234480    0.063423               30                2               2  2022-03-21T04:49:00+08:00      44.757558       0.062693
4           frontend-0                       trace.from_root.row_count   43.550000   31.450000               30                2               2  2022-03-21T04:34:00+08:00      48.000000      18.000000
5           frontend-0                        trace.self.duration_mean    0.035291    0.004563               30                2               2  2022-03-21T04:41:00+08:00       0.678842       0.004445
6           frontend-0                         trace.self.duration_p95    0.008336    0.007588               30                2               2  2022-03-21T04:41:00+08:00       0.008589       0.007496
7           frontend-0                            trace.self.row_count  444.850000  316.200000               30                2               2  2022-03-21T04:34:00+08:00     478.000000     151.000000
8           frontend-0               trace.to_adservice-0.duration_p95    0.000072    0.000019               30                2               2  2022-03-21T04:31:00+08:00       0.000078       0.000016
9           frontend-0                trace.to_cartservice-2.row_count   13.100000    9.450000               30                2               2  2022-03-21T04:34:00+08:00      14.000000       5.000000
10          frontend-0        trace.to_currencyservice-0.duration_mean    0.000144    0.000111               30                2               2  2022-03-21T04:41:00+08:00       0.000154       0.000110
11          frontend-0         trace.to_currencyservice-0.duration_p95    0.000164    0.000132               30                2               2  2022-03-21T04:36:00+08:00       0.000267       0.000131
12          frontend-0            trace.to_currencyservice-0.row_count   47.100000   26.800000               30                2               2  2022-03-21T04:34:00+08:00      51.000000      17.000000
13          frontend-0        trace.to_currencyservice-1.duration_mean    0.000141    0.000116               30                2               2  2022-03-21T04:40:00+08:00       0.000144       0.000115
14          frontend-0         trace.to_currencyservice-1.duration_p95    0.000167    0.000137               30                2               2  2022-03-21T04:32:00+08:00       0.000182       0.000136
15          frontend-0            trace.to_currencyservice-1.row_count   47.100000   27.350000               30                2               2  2022-03-21T04:34:00+08:00      50.000000      17.000000
16          frontend-0        trace.to_currencyservice-2.duration_mean    0.000155    0.000122               30                2               2  2022-03-21T04:30:00+08:00       0.000169       0.000122
17          frontend-0            trace.to_currencyservice-2.row_count   47.100000   26.800000               30                2               2  2022-03-21T04:34:00+08:00      51.000000      18.000000
18          frontend-0  trace.to_productcatalogservice-0.duration_mean    0.004920    0.004546               30                2               2  2022-03-21T04:32:00+08:00       0.004926       0.004410
19          frontend-0   trace.to_productcatalogservice-0.duration_p95    0.006361    0.005702               30                2               2  2022-03-21T04:36:00+08:00       0.006448       0.005627```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing log_df / error_logs_df if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

# Normalize timestamps to tz-aware datetimes in Asia/Shanghai
if log_df['timestamp'].dtype.kind in {'i', 'u', 'f'}:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], utc=True).dt.tz_convert(tz)

log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) Global P95/P5 per (cmdb_id, log_name) using full series
group_cols = ['cmdb_id', 'log_name']
global_stats = log_df.groupby(group_cols)['value'].agg(
    global_P95=lambda s: s.quantile(0.95),
    global_P5=lambda s: s.quantile(0.05)
).reset_index()

# Incident window (inclusive)
start = tz.localize(pd.to_datetime("2022-03-21 04:30:00"))
end = tz.localize(pd.to_datetime("2022-03-21 05:00:00"))

# Filter log rows to window
log_window = log_df[(log_df['ts'] >= start) & (log_df['ts'] <= end)].copy()

# Merge thresholds into window rows
window_with_thresholds = log_window.merge(global_stats, on=group_cols, how='left')

# Compute anomaly flags
window_with_thresholds['is_above_p95'] = window_with_thresholds['value'] > window_with_thresholds['global_P95']
window_with_thresholds['is_below_p5'] = window_with_thresholds['value'] < window_with_thresholds['global_P5']
window_with_thresholds['anomaly_ts'] = window_with_thresholds['ts'].where(window_with_thresholds['is_above_p95'] | window_with_thresholds['is_below_p5'])

# Aggregate per pair within window
agg_window = window_with_thresholds.groupby(group_cols).agg(
    count_in_window=('value', 'count'),
    count_above_p95=('is_above_p95', 'sum'),
    count_below_p5=('is_below_p5', 'sum'),
    earliest_anom_ts=('anomaly_ts', 'min'),
    max_in_window=('value', 'max'),
    min_in_window=('value', 'min')
).reset_index()

# Merge global stats with window aggregates to include pairs without window rows
log_pair_table = global_stats.merge(agg_window, on=group_cols, how='left')

# Fill missing counts with zeros
log_pair_table['count_in_window'] = log_pair_table['count_in_window'].fillna(0).astype(int)
log_pair_table['count_above_p95'] = log_pair_table['count_above_p95'].fillna(0).astype(int)
log_pair_table['count_below_p5'] = log_pair_table['count_below_p5'].fillna(0).astype(int)

# earliest_anom_ts to ISO strings or None
def to_iso(x):
    if pd.isna(x):
        return None
    if x.tzinfo is None:
        x = tz.localize(x)
    return x.isoformat()

log_pair_table['earliest_anom_ts'] = log_pair_table['earliest_anom_ts'].apply(to_iso)

# Compute anomaly severity key and sort before selecting final columns
log_pair_table['anom_count_max'] = log_pair_table[['count_above_p95', 'count_below_p5']].max(axis=1)
sorted_logs = log_pair_table.sort_values(by=['anom_count_max', 'count_in_window'], ascending=[False, False])

# Prepare result columns and limit to top 20
result_cols = [
    'cmdb_id', 'log_name', 'global_P95', 'global_P5',
    'count_in_window', 'count_above_p95', 'count_below_p5',
    'earliest_anom_ts', 'max_in_window', 'min_in_window'
]
log_anom_top = sorted_logs[result_cols].head(20).reset_index(drop=True)

# 2) Extract error_logs rows within the same incident window (store full filtered set)
if error_logs_df['timestamp'].dtype.kind in {'i', 'u', 'f'}:
    error_logs_df['ts'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    error_logs_df['ts'] = pd.to_datetime(error_logs_df['timestamp'], utc=True).dt.tz_convert(tz)

error_logs_window = error_logs_df[(error_logs_df['ts'] >= start) & (error_logs_df['ts'] <= end)].copy()
error_logs_window['ts_iso'] = error_logs_window['ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)

# Return compact results: top log anomalies and up to 20 error log rows
log_anom_top, error_logs_window.head(20)

```
Out[5]:
```
```python
# Compose a concise plain-English summary based on log_anom_top and error_logs_window in the kernel.
try:
    log_anom_top
    error_logs_window
except NameError:
    raise RuntimeError("Expected variables log_anom_top and error_logs_window in kernel.")

summary = (
    "Summary of log and error-log findings for the incident window (2022-03-21 04:30–05:00 +08):\n"
    "- Log anomalies: The top anomalous series are primarily the 'log.row_count' metric across many services\n"
    "  (examples: cartservice-0/2, currencyservice-1/2, frontend-0/1/2, productcatalogservice-*, shippingservice-*).\n"
    "  Most of these pairs have full-window coverage (30–31 points) and show small but consistent excursions\n"
    "  — typically 2 samples above the series' global P95 and 2 samples below the global P5 during the window.\n"
    "  A few series have slightly different splits (e.g., recommendationservice-2 shows 1 above / 2 below).\n"
    "  Interpretation: request-rate / row-count fluctuations affected many services concurrently and should be\n"
    "  investigated as a potential upstream traffic or frontend-related disturbance.\n"
    "- Error logs: There is one error-log entry in the window (shown):\n"
    "  timestamp 2022-03-21T04:57:13+08, cmdb_id frontend-2, message 'severity: error, message: request error'.\n"
    "  This single error from frontend-2 is a concrete symptom to investigate alongside the widespread log.row_count anomalies.\n"
    "- Recommended next steps: prioritize investigating frontend components (notably frontend-2) and traffic/load\n"
    "  sources that could explain simultaneous row_count excursions across many services."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id       log_name  global_P95  global_P5  count_in_window  count_above_p95  count_below_p5           earliest_anom_ts  max_in_window  min_in_window
0             cartservice-0  log.row_count      309.00     205.50               31                2               2  2022-03-21T04:30:00+08:00          330.0            9.0
1             cartservice-2  log.row_count      310.50     206.00               31                2               2  2022-03-21T04:34:00+08:00          330.0            9.0
2         currencyservice-1  log.row_count      290.00     166.50               31                2               2  2022-03-21T04:34:00+08:00          321.0            5.0
3         currencyservice-2  log.row_count      287.50     166.00               31                2               2  2022-03-21T04:34:00+08:00          316.0            2.0
4                frontend-1  log.row_count      687.50     401.50               31                2               2  2022-03-21T04:30:00+08:00          731.0            4.0
5   productcatalogservice-2  log.row_count      372.50     245.50               31                2               2  2022-03-21T04:34:00+08:00          381.0           65.0
6   recommendationservice-2  log.row_count       83.00      47.50               31                1               2  2022-03-21T04:37:00+08:00           86.0            1.0
7         shippingservice-1  log.row_count       41.50      18.00               31                2               2  2022-03-21T04:34:00+08:00           47.0            2.0
8         shippingservice-2  log.row_count       42.00      20.00               31                2               2  2022-03-21T04:36:00+08:00           48.0            2.0
9               adservice-0  log.row_count       52.00      38.90               30                0               2  2022-03-21T04:45:00+08:00           52.0           32.0
10              adservice-1  log.row_count       39.10      28.90               30                2               2  2022-03-21T04:37:00+08:00           42.0           16.0
11            cartservice-1  log.row_count      324.75     240.00               30                2               1  2022-03-21T04:37:00+08:00          328.0          171.0
12        currencyservice-0  log.row_count      286.20     193.95               30                2               2  2022-03-21T04:34:00+08:00          315.0          143.0
13               frontend-0  log.row_count      683.45     487.90               30                2               2  2022-03-21T04:34:00+08:00          732.0          259.0
14               frontend-2  log.row_count      263.25     107.00               30                2               2  2022-03-21T04:40:00+08:00          280.0           50.0
15  productcatalogservice-0  log.row_count      378.20     283.50               30                2               2  2022-03-21T04:34:00+08:00          385.0          219.0
16  productcatalogservice-1  log.row_count      372.65     281.55               30                2               2  2022-03-21T04:34:00+08:00          384.0          219.0
17  recommendationservice-1  log.row_count       81.55      66.00               30                2               1  2022-03-21T04:30:00+08:00           86.0           52.0
18        shippingservice-0  log.row_count       41.10      16.00               30                2               1  2022-03-21T04:34:00+08:00           45.0           13.0
19        checkoutservice-1  log.row_count       35.95       3.00               28                2               1  2022-03-21T04:32:00+08:00           51.0            1.0,     timestamp     cmdb_id                                  message                        ts                     ts_iso
0  1647809833  frontend-2  severity: error, message: request error 2022-03-21 04:57:13+08:00  2022-03-21T04:57:13+08:00)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp column converted to tz-aware datetime in Asia/Shanghai
if 'ts' not in metric_df.columns:
    if metric_df['timestamp'].dtype.kind in {'i', 'u', 'f'}:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], utc=True).dt.tz_convert(tz)
else:
    # ensure tz
    metric_df['ts'] = pd.to_datetime(metric_df['ts'], utc=True).dt.tz_convert(tz)

metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Focus on cmdb_id 'frontend-2'
cmdb = 'frontend-2'
metric_f2 = metric_df[metric_df['cmdb_id'] == cmdb].copy()

# Compute global P95 and P5 per kpi_name using full available series (rule 9)
global_stats_f2 = metric_f2.groupby('kpi_name')['value'].agg(
    global_P95=lambda s: s.quantile(0.95),
    global_P5=lambda s: s.quantile(0.05)
).reset_index()

# Incident window (inclusive)
start = tz.localize(pd.to_datetime("2022-03-21 04:30:00"))
end = tz.localize(pd.to_datetime("2022-03-21 05:00:00"))

# Filter to window
metric_f2_window = metric_f2[(metric_f2['ts'] >= start) & (metric_f2['ts'] <= end)].copy()

# Merge global thresholds into in-window points
metric_f2_window = metric_f2_window.merge(global_stats_f2, on='kpi_name', how='left')

# Mark anomaly points
metric_f2_window['breach_type'] = np.where(
    metric_f2_window['value'] > metric_f2_window['global_P95'], 'high',
    np.where(metric_f2_window['value'] < metric_f2_window['global_P5'], 'low', None)
)

# Keep only anomalous points
anom_pts = metric_f2_window[metric_f2_window['breach_type'].notna()].copy()
# If no anomalies, produce empty result
if anom_pts.empty:
    frontend2_fault_segments = pd.DataFrame(columns=[
        'kpi_name','breach_type','start_ts','end_ts','duration_minutes','count_points',
        'max_in_segment','min_in_segment','global_P95','global_P5','severity_ratio'
    ])
else:
    # Sort for grouping
    anom_pts = anom_pts.sort_values(['kpi_name','breach_type','ts']).reset_index(drop=True)

    # For grouping consecutive-minute segments within each (kpi_name, breach_type)
    def assign_segment_ids(df):
        df = df.sort_values('ts').copy()
        # compute diff in seconds
        df['diff_s'] = df['ts'].diff().dt.total_seconds().fillna(0)
        # start new segment when diff_s != 60
        df['new_segment'] = (df['diff_s'] != 60).astype(int)
        df['segment_id_local'] = df['new_segment'].cumsum()
        return df

    anom_pts = anom_pts.groupby(['kpi_name','breach_type'], group_keys=False).apply(assign_segment_ids)
    # Create a global segment id combining kpi, breach_type, segment_id_local
    anom_pts['segment_id'] = anom_pts.apply(lambda r: f"{r['kpi_name']}|{r['breach_type']}|{int(r['segment_id_local'])}", axis=1)

    # Aggregate per segment
    seg_agg = anom_pts.groupby('segment_id').agg(
        kpi_name=('kpi_name','first'),
        breach_type=('breach_type','first'),
        start_ts=('ts','min'),
        end_ts=('ts','max'),
        count_points=('ts','count'),
        max_in_segment=('value','max'),
        min_in_segment=('value','min'),
        global_P95=('global_P95','first'),
        global_P5=('global_P5','first'),
    ).reset_index()

    # Duration in minutes: inclusive span
    seg_agg['duration_minutes'] = ((seg_agg['end_ts'] - seg_agg['start_ts']).dt.total_seconds() / 60).astype(int) + 1
    # Keep only segments with count_points >= 2 (discard isolated single-point anomalies)
    seg_agg = seg_agg[seg_agg['count_points'] >= 2].copy()

    # Compute severity_ratio per segment
    # For 'high': max(|value - global_P95|/global_P95) within the segment
    # For 'low':  max(|global_P5 - value|/global_P5)
    def compute_severity(row):
        seg_id = row['segment_id']
        rows = anom_pts[anom_pts['segment_id'] == seg_id]
        if row['breach_type'] == 'high':
            denom = row['global_P95']
            if pd.isna(denom) or denom == 0:
                return np.nan
            vals = (rows['value'] - denom).abs() / denom
            return float(vals.max())
        else:
            denom = row['global_P5']
            if pd.isna(denom) or denom == 0:
                return np.nan
            vals = (row['global_P5'] - rows['value']).abs() / denom
            return float(vals.max())

    seg_agg['severity_ratio'] = seg_agg.apply(compute_severity, axis=1)

    # Convert start_ts and end_ts to ISO strings
    seg_agg['start_ts'] = seg_agg['start_ts'].apply(lambda x: x.isoformat())
    seg_agg['end_ts'] = seg_agg['end_ts'].apply(lambda x: x.isoformat())

    # Select and order columns as requested
    frontend2_fault_segments = seg_agg[[
        'kpi_name','breach_type','start_ts','end_ts','duration_minutes','count_points',
        'max_in_segment','min_in_segment','global_P95','global_P5','severity_ratio'
    ]].copy()

    # Order by severity_ratio desc then duration_minutes desc
    frontend2_fault_segments = frontend2_fault_segments.sort_values(
        by=['severity_ratio','duration_minutes'], ascending=[False, False]
    ).reset_index(drop=True)

# Keep only top 20 for display as required, but store full table in kernel variable frontend2_fault_segments
frontend2_fault_segments = frontend2_fault_segments.reset_index(drop=True)
frontend2_fault_segments.head(20)

```
Out[6]:
```
```python
# Compose a concise plain-English summary based on frontend2_fault_segments in the kernel.
try:
    frontend2_fault_segments
except NameError:
    raise RuntimeError("frontend2_fault_segments not found in kernel. Please run the segment detection step first.")

df = frontend2_fault_segments.copy()

total_segments = len(df)
# Top segment by severity_ratio (df is already sorted by severity_ratio desc)
top = df.iloc[0] if total_segments > 0 else None

# Count of high vs low breach segments
counts_by_type = df['breach_type'].value_counts().to_dict()

# Examples: top 6 segments (kpi :: breach_type @ start_ts)
examples = []
for _, r in df.head(6).iterrows():
    examples.append(f"{r['kpi_name']} :: {r['breach_type']} @ {r['start_ts']}–{r['end_ts']} (pts={int(r['count_points'])}, sev={r['severity_ratio']:.3f})")

summary_lines = []
summary_lines.append(f"frontend-2 anomaly segment summary (incident window 2022-03-21 04:30–05:00 +08):")
summary_lines.append(f"- Detected segments (count after removing isolated single-point anomalies): {total_segments}.")
summary_lines.append(f"- Breach types: {counts_by_type}. Most segments are 2-minute consecutive breaches.")
if top is not None:
    summary_lines.append(
        f"- Top segment by severity: {top['kpi_name']} (breach_type={top['breach_type']}) from {top['start_ts']} to {top['end_ts']}, "
        f"duration={int(top['duration_minutes'])} min, points={int(top['count_points'])}, severity_ratio={float(top['severity_ratio']):.3f}."
    )
summary_lines.append("- Notable pattern:")
summary_lines.append(
    "  • A very large relative high breach was observed for mesh.source.frontend.recommendationservice.* at 04:55–04:56 (+40x relative to P95)."
)
summary_lines.append(
    "  • Many low-type segments cluster around 04:57–04:58 across multiple mesh.source.frontend.* series (consistent 2-minute drops relative to their P5s, severity ratios ~0.8)."
)
summary_lines.append("- Implication: frontend-2 shows both a short but extremely severe high spike (recommendationservice path) and widespread short low dips "
                     "on multiple request/response mesh metrics near 04:57. Investigate frontend outbound calls, recommendationservice interactions, and network/mesh behavior.")
summary_lines.append("- Examples of top segments:")
summary_lines.extend("  • " + e for e in examples)

summary_text = "\n".join(summary_lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name breach_type                   start_ts                     end_ts  duration_minutes  count_points  max_in_segment  min_in_segment     global_P95      global_P5  severity_ratio
0   mesh.source.frontend.recommendationservice.ist...        high  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2        2.600900        0.233325       0.062225       0.029350       40.798313
1   container.node-5.container_network_receive_MB....        high  2022-03-21T04:41:00+08:00  2022-03-21T04:42:00+08:00                 2             2        0.358744        0.357589       0.107671       0.051962        2.331847
2   mesh.destination.unknown.frontend.istio_respon...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2    65150.000000     8300.000000  158975.000000   77925.000000        0.893487
3   mesh.destination.unknown.frontend.istio_reques...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2     1750.000000      335.000000    5132.500000    2890.000000        0.884083
4   mesh.source.frontend.productcatalogservice.ist...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2     9635.000000     2050.000000   26467.500000   15192.500000        0.865065
5   mesh.source.frontend.recommendationservice.ist...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2     7750.000000     1550.000000   19375.000000   10075.000000        0.846154
6   mesh.source.frontend.recommendationservice.ist...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2     1375.000000      275.000000    3437.500000    1787.500000        0.846154
7   mesh.source.frontend.productcatalogservice.ist...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2    51150.000000    10850.000000  127100.000000   66650.000000        0.837209
8   mesh.source.frontend.productcatalogservice.ist...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2        0.190150        0.039350       0.493725       0.241225        0.836874
9   mesh.source.frontend.currencyservice.istio_req...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2    22475.000000     5425.000000   85250.000000   32550.000000        0.833333
10  mesh.source.frontend.currencyservice.istio_req...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2        0.023225        0.005675       0.080925       0.033550        0.830849
11  mesh.source.frontend.currencyservice.istio_req...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2       16.000000        3.500000      55.500000      20.500000        0.829268
12  mesh.source.frontend.currencyservice.istio_req...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2       16.000000        3.500000      55.500000      20.500000        0.829268
13  mesh.source.frontend.currencyservice.istio_res...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2       16.000000        3.500000      55.500000      20.500000        0.829268
14  mesh.source.frontend.jaeger-collector.istio_re...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2    81300.000000    21725.000000  221750.000000  123050.000000        0.823446
15  mesh.source.frontend.currencyservice.istio_res...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2     3937.500000     1087.500000   14475.000000    5925.000000        0.816456
16  mesh.destination.unknown.frontend.istio_reques...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2        5.500000        1.500000      14.500000       8.000000        0.812500
17  mesh.source.frontend.jaeger-collector.istio_re...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2     1094.500000      348.250000    2835.750000    1592.000000        0.781250
18  mesh.source.frontend.productcatalogservice.ist...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2       36.500000        9.500000      79.000000      41.500000        0.771084
19  mesh.source.frontend.productcatalogservice.ist...         low  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2       36.500000        9.500000      79.000000      41.500000        0.771084```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure ts column is present and timezone-aware in Asia/Shanghai
if 'ts' not in metric_df.columns:
    if metric_df['timestamp'].dtype.kind in {'i','u','f'}:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], utc=True).dt.tz_convert(tz)
else:
    metric_df['ts'] = pd.to_datetime(metric_df['ts'], utc=True).dt.tz_convert(tz)

metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Target recommendation service cmdb_ids
targets = ['recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0']

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full available series
df_targets_all = metric_df[metric_df['cmdb_id'].isin(targets)].copy()
group_cols = ['cmdb_id', 'kpi_name']
global_stats = df_targets_all.groupby(group_cols)['value'].agg(
    global_P95=lambda s: s.quantile(0.95),
    global_P5=lambda s: s.quantile(0.05)
).reset_index()

# 2) Filter metric rows to the incident window (inclusive)
start = tz.localize(pd.to_datetime("2022-03-21 04:30:00"))
end = tz.localize(pd.to_datetime("2022-03-21 05:00:00"))
window_df = metric_df[
    (metric_df['cmdb_id'].isin(targets)) &
    (metric_df['ts'] >= start) &
    (metric_df['ts'] <= end)
].copy()

# 3) Merge global thresholds into window rows and compute anomaly flags
window_with_thresholds = window_df.merge(global_stats, on=group_cols, how='left')
window_with_thresholds['is_above_p95'] = window_with_thresholds['value'] > window_with_thresholds['global_P95']
window_with_thresholds['is_below_p5'] = window_with_thresholds['value'] < window_with_thresholds['global_P5']
window_with_thresholds['anomaly_ts'] = window_with_thresholds['ts'].where(window_with_thresholds['is_above_p95'] | window_with_thresholds['is_below_p5'])

# 4) Aggregate per (cmdb_id, kpi_name) within the window
agg_window = window_with_thresholds.groupby(group_cols).agg(
    count_in_window=('value','count'),
    count_above_p95=('is_above_p95','sum'),
    count_below_p5=('is_below_p5','sum'),
    earliest_anom_ts=('anomaly_ts','min'),
    max_in_window=('value','max'),
    min_in_window=('value','min')
).reset_index()

# 5) Ensure pairs with no window rows are included by merging global_stats with agg_window
pair_table = global_stats.merge(agg_window, on=group_cols, how='left')

# Fill missing counts with zeros and format earliest_anom_ts
pair_table['count_in_window'] = pair_table['count_in_window'].fillna(0).astype(int)
pair_table['count_above_p95'] = pair_table['count_above_p95'].fillna(0).astype(int)
pair_table['count_below_p5'] = pair_table['count_below_p5'].fillna(0).astype(int)

def to_iso(x):
    if pd.isna(x):
        return None
    if x.tzinfo is None:
        x = tz.localize(x)
    return x.isoformat()

pair_table['earliest_anom_ts'] = pair_table['earliest_anom_ts'].apply(to_iso)

# 6) Select requested columns and order by max(count_above_p95, count_below_p5) desc
pair_table['anom_count_max'] = pair_table[['count_above_p95','count_below_p5']].max(axis=1)
result_cols = [
    'cmdb_id','kpi_name','global_P95','global_P5',
    'count_in_window','count_above_p95','count_below_p5',
    'earliest_anom_ts','max_in_window','min_in_window'
]
recsvc_anom_check = pair_table.sort_values(
    by=['anom_count_max','count_in_window'], ascending=[False, False]
)[result_cols].head(20).reset_index(drop=True)

recsvc_anom_check

```
Out[7]:
```
```python
# Compose a concise plain-English summary using recsvc_anom_check from the kernel.
try:
    recsvc_anom_check
except NameError:
    raise RuntimeError("recsvc_anom_check not found in kernel. Please run the anomaly check step first.")

df = recsvc_anom_check.copy()

# Key observations
cmdbs = df['cmdb_id'].unique().tolist()
total_rows = df.shape[0]
earliest = df['earliest_anom_ts'].dropna().min()
latest = df['earliest_anom_ts'].dropna().max()

# Summarize top issues by quick inspection
top_examples = df.head(6).apply(
    lambda r: f"{r['cmdb_id']} :: {r['kpi_name']} (above P95={int(r['count_above_p95'])}, below P5={int(r['count_below_p5'])}, first={r['earliest_anom_ts']})",
    axis=1
).tolist()

summary_lines = [
    "Recommendation service metric anomaly summary (recommendationservice-0/1/2, recommendationservice2-0):",
    f"- Components checked: {', '.join(cmdbs)} (total metric pairs returned: {total_rows}).",
    f"- Anomalies are widespread across CPU, filesystem usage, network (tx/rx), and mesh/jaeger-related KPIs.",
    f"- Earliest anomaly timestamps in these results range ~ {earliest} to {latest}.",
    "- Representative top metric anomalies (examples):"
]
summary_lines += [f"  • {e}" for e in top_examples]
summary_lines += [
    "- Patterns to note:",
    "  • Many KPI series show full-window data (31 points) with multiple samples above P95 and/or below P5 — not single isolated blips.",
    "  • Filesystem usage and container 'last_seen' show multiple above-P95 events clustered at ~04:58, suggesting activity/updates near that time.",
    "  • CPU-related metrics and CPU throttling (cfs_periods) show multi-sample anomalies around ~04:33–04:36 on several instances.",
    "- Interpretation and next steps:",
    "  • The recommendationservice pods exhibit concurrent anomalies across resource (CPU, FS), network, and tracing metrics.",
    "  • Investigate pod/container-level events (restarts, last_seen updates), node resource contention, and network/mesh behavior around 04:30–05:00.",
    "  • Correlate with logs and container lifecycle events (kube events) for these pods to pinpoint root cause."
]

summary_text = "\n".join(summary_lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_P95     global_P5  count_in_window  count_above_p95  count_below_p5           earliest_anom_ts  max_in_window  min_in_window
0   recommendationservice-0       container.node-6.container_cpu_usage_seconds  1.003225e-01  7.496938e-02               31                1               3  2022-03-21T04:43:00+08:00   1.017208e-01   6.874102e-02
1   recommendationservice-0   container.node-6.container_fs_usage_MB./dev/vda1  4.146875e+01  4.114453e+01               31                3               0  2022-03-21T04:58:00+08:00   4.148438e+01   4.130859e+01
2   recommendationservice-0               container.node-6.container_last_seen  1.647810e+09  1.647807e+09               31                3               0  2022-03-21T04:58:00+08:00   1.647810e+09   1.647808e+09
3   recommendationservice-0  container.node-6.container_network_receive_MB....  8.820486e-02  6.558895e-02               31                1               3  2022-03-21T04:32:00+08:00   6.449876e-01   6.176901e-02
4   recommendationservice-0  container.node-6.container_network_transmit_MB...  1.074858e+00  1.057258e+00               31                1               3  2022-03-21T04:32:00+08:00   1.093955e+00   1.056020e+00
5   recommendationservice-0  container.node-6.container_network_transmit_pa...  3.590000e+02  2.880000e+02               31                1               3  2022-03-21T04:32:00+08:00   4.280000e+02   2.820000e+02
6   recommendationservice-0  mesh.destination.frontend.recommendationservic...  8.012500e-02  6.192500e-02               31                0               3  2022-03-21T04:33:00+08:00   7.770000e-02   4.362500e-02
7   recommendationservice-0  mesh.source.recommendationservice.jaeger-colle...  3.345000e+04  2.842500e+04               31                2               3  2022-03-21T04:35:00+08:00   3.370000e+04   2.290000e+04
8   recommendationservice-1         container.node-6.container_cpu_cfs_periods  1.055000e+02  7.550000e+01               31                3               2  2022-03-21T04:36:00+08:00   1.110000e+02   6.700000e+01
9   recommendationservice-1       container.node-6.container_cpu_usage_seconds  1.010108e-01  7.876514e-02               31                3               3  2022-03-21T04:36:00+08:00   1.087035e-01   6.957096e-02
10  recommendationservice-1   container.node-6.container_fs_usage_MB./dev/vda1  4.146094e+01  4.113477e+01               31                3               0  2022-03-21T04:58:00+08:00   4.148047e+01   4.130078e+01
11  recommendationservice-1               container.node-6.container_last_seen  1.647810e+09  1.647807e+09               31                3               0  2022-03-21T04:58:00+08:00   1.647810e+09   1.647808e+09
12  recommendationservice-1  mesh.destination.frontend.recommendationservic...  8.515000e-02  6.037500e-02               31                0               3  2022-03-21T04:44:00+08:00   8.215000e-02   5.175000e-02
13  recommendationservice-1  mesh.source.recommendationservice.jaeger-colle...  2.100000e-03  0.000000e+00               31                3               0  2022-03-21T04:30:00+08:00   3.550000e-03   0.000000e+00
14  recommendationservice-2         container.node-6.container_cpu_cfs_periods  1.565000e+02  1.175000e+02               31                1               3  2022-03-21T04:33:00+08:00   1.735000e+02   1.110000e+02
15  recommendationservice-2       container.node-6.container_cpu_usage_seconds  1.396928e-01  9.816999e-02               31                1               3  2022-03-21T04:33:00+08:00   1.440732e-01   8.587756e-02
16  recommendationservice-2        container.node-6.container_cpu_user_seconds  9.000000e-02  5.500000e-02               31                1               3  2022-03-21T04:44:00+08:00   1.050000e-01   5.000000e-02
17  recommendationservice-2   container.node-6.container_fs_usage_MB./dev/vda1  4.145703e+01  4.113477e+01               31                3               0  2022-03-21T04:58:00+08:00   4.148047e+01   4.129688e+01
18  recommendationservice-2               container.node-6.container_last_seen  1.647810e+09  1.647807e+09               31                3               0  2022-03-21T04:58:00+08:00   1.647810e+09   1.647808e+09
19  recommendationservice-2  container.node-6.container_network_transmit_MB...  1.071615e+00  1.047930e+00               31                1               3  2022-03-21T04:35:00+08:00   1.111681e+00   1.019731e+00```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure ts column with Asia/Shanghai tz
if 'ts' not in metric_df.columns:
    if metric_df['timestamp'].dtype.kind in {'i','u','f'}:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], utc=True).dt.tz_convert(tz)
else:
    metric_df['ts'] = pd.to_datetime(metric_df['ts'], utc=True).dt.tz_convert(tz)

metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Target recommendation service pods
targets = ['recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0']

# Subset full series for targets to compute global thresholds (rule 9)
target_full = metric_df[metric_df['cmdb_id'].isin(targets)].copy()

group_cols = ['cmdb_id','kpi_name']
global_stats = target_full.groupby(group_cols)['value'].agg(
    global_P95=lambda s: s.quantile(0.95),
    global_P5=lambda s: s.quantile(0.05)
).reset_index()

# Incident window (inclusive)
start = tz.localize(pd.to_datetime("2022-03-21 04:30:00"))
end = tz.localize(pd.to_datetime("2022-03-21 05:00:00"))

# Filter points to window and merge thresholds
window_df = metric_df[
    (metric_df['cmdb_id'].isin(targets)) &
    (metric_df['ts'] >= start) &
    (metric_df['ts'] <= end)
].copy()

window_df = window_df.merge(global_stats, on=group_cols, how='left')

# Mark breach types
window_df['breach_type'] = np.where(
    window_df['value'] > window_df['global_P95'], 'high',
    np.where(window_df['value'] < window_df['global_P5'], 'low', None)
)

# Keep only anomalous points
anom_pts = window_df[window_df['breach_type'].notna()].copy()

if anom_pts.empty:
    # create empty dataframe with requested columns
    recsvc_fault_segments = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','breach_type','start_ts','end_ts','duration_minutes','count_points',
        'max_in_segment','min_in_segment','global_P95','global_P5','severity_ratio'
    ])
else:
    # Sort for grouping
    anom_pts = anom_pts.sort_values(['cmdb_id','kpi_name','breach_type','ts']).reset_index(drop=True)

    # Assign local segment ids per (cmdb_id,kpi_name,breach_type) when timestamps are not consecutive by 60s
    def assign_local_segments(df):
        df = df.sort_values('ts').copy()
        df['diff_s'] = df['ts'].diff().dt.total_seconds().fillna(0)
        df['new_seg'] = (df['diff_s'] != 60).astype(int)
        df['seg_local_id'] = df['new_seg'].cumsum()
        return df

    anom_pts = anom_pts.groupby(['cmdb_id','kpi_name','breach_type'], group_keys=False).apply(assign_local_segments).reset_index(drop=True)

    # Create a global segment id
    anom_pts['segment_id'] = anom_pts.apply(lambda r: f"{r['cmdb_id']}|{r['kpi_name']}|{r['breach_type']}|{int(r['seg_local_id'])}", axis=1)

    # Aggregate per segment
    seg_agg = anom_pts.groupby('segment_id').agg(
        cmdb_id=('cmdb_id','first'),
        kpi_name=('kpi_name','first'),
        breach_type=('breach_type','first'),
        start_ts=('ts','min'),
        end_ts=('ts','max'),
        count_points=('ts','count'),
        max_in_segment=('value','max'),
        min_in_segment=('value','min'),
        global_P95=('global_P95','first'),
        global_P5=('global_P5','first')
    ).reset_index()

    # Duration minutes (inclusive)
    seg_agg['duration_minutes'] = ((seg_agg['end_ts'] - seg_agg['start_ts']).dt.total_seconds() / 60).astype(int) + 1

    # Discard single-point segments
    seg_agg = seg_agg[seg_agg['count_points'] >= 2].copy()

    # Compute severity_ratio per segment by scanning anom_pts rows for that segment
    def compute_severity(row):
        sid = row['segment_id']
        rows = anom_pts[anom_pts['segment_id'] == sid]
        if row['breach_type'] == 'high':
            denom = row['global_P95']
            if pd.isna(denom) or denom == 0:
                return np.nan
            vals = (rows['value'] - denom).abs() / denom
            return float(vals.max())
        else:
            denom = row['global_P5']
            if pd.isna(denom) or denom == 0:
                return np.nan
            vals = (row['global_P5'] - rows['value']).abs() / denom
            return float(vals.max())

    seg_agg['severity_ratio'] = seg_agg.apply(compute_severity, axis=1)

    # Convert timestamps to ISO strings
    seg_agg['start_ts'] = seg_agg['start_ts'].apply(lambda x: x.isoformat())
    seg_agg['end_ts'] = seg_agg['end_ts'].apply(lambda x: x.isoformat())

    # Select requested columns and sort
    recsvc_fault_segments = seg_agg[[
        'cmdb_id','kpi_name','breach_type','start_ts','end_ts','duration_minutes','count_points',
        'max_in_segment','min_in_segment','global_P95','global_P5','severity_ratio'
    ]].copy()

    recsvc_fault_segments = recsvc_fault_segments.sort_values(by=['severity_ratio','duration_minutes'], ascending=[False, False]).reset_index(drop=True)

# Keep only top 20 for display but store full table in kernel variable recsvc_fault_segments
recsvc_fault_segments = recsvc_fault_segments.reset_index(drop=True)
recsvc_fault_segments.head(20)

```
Out[8]:
```
```python
# Compose a concise plain-English summary from recsvc_fault_segments stored in the kernel.
try:
    recsvc_fault_segments
except NameError:
    raise RuntimeError("recsvc_fault_segments not found in kernel. Please run the segmentation step first.")

df = recsvc_fault_segments.copy()

total_segments = len(df)
by_cmdb = df['cmdb_id'].value_counts().to_dict()
time_min = df['start_ts'].min() if total_segments>0 else "N/A"
time_max = df['end_ts'].max() if total_segments>0 else "N/A"
top = df.iloc[0] if total_segments>0 else None

examples = []
for _, r in df.head(5).iterrows():
    examples.append(f"{r['cmdb_id']} :: {r['kpi_name']} ({r['breach_type']}) {r['start_ts']}–{r['end_ts']}, pts={int(r['count_points'])}, sev={r['severity_ratio']:.3f}")

summary = (
    f"Recommendation-service fault-segmentation summary:\n"
    f"- Total multi-point anomalous segments detected: {total_segments}.\n"
    f"- Breakdown by pod (top counts): {by_cmdb}.\n"
    f"- Time window of segments: {time_min} to {time_max} (incident window 04:30–05:00 +08).\n"
    f"- Highest-severity segment: "
    + (f"{top['cmdb_id']} :: {top['kpi_name']} ({top['breach_type']}) {top['start_ts']}–{top['end_ts']}, "
       f"pts={int(top['count_points'])}, severity_ratio={top['severity_ratio']:.3f}"
       if top is not None else "N/A") + "\n"
    f"- Patterns observed: many segments cluster around 04:55–04:58; recommendationservice-2 shows the most and the largest relative breaches (both extreme highs and lows), "
    "including very large high spikes on jaeger/mesh related KPIs and multiple low drops on mesh/response/request metrics.\n"
    f"- Examples (top 5):\n  " + "\n  ".join(examples) + "\n"
    f"- Actionable note: prioritize investigation of recommendationservice-2 (04:55–04:58), its mesh interactions and jaeger collector traces, "
    "and related network/container events to find root cause."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name breach_type                   start_ts                     end_ts  duration_minutes  count_points  max_in_segment  min_in_segment    global_P95     global_P5  severity_ratio
0    recommendationservice-2  mesh.source.recommendationservice.jaeger-colle...        high  2022-03-21T04:57:00+08:00  2022-03-21T04:58:00+08:00                 2             2      109.225000        23.57710      0.001050      0.000000   104022.809524
1    recommendationservice-2  mesh.source.recommendationservice.productcatal...        high  2022-03-21T04:56:00+08:00  2022-03-21T04:58:00+08:00                 3             3       93.519200        12.02860      5.507125      0.019950       15.981492
2    recommendationservice-2  mesh.destination.frontend.recommendationservic...        high  2022-03-21T04:56:00+08:00  2022-03-21T04:58:00+08:00                 3             3       93.538200        12.06010      5.540625      0.065425       15.882247
3    recommendationservice-2  mesh.source.recommendationservice.jaeger-colle...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:57:00+08:00                 3             3    16850.000000         0.00000  33400.000000  25375.000000        1.000000
4    recommendationservice-2  mesh.source.recommendationservice.jaeger-colle...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:57:00+08:00                 3             3        5.000000         0.00000     12.000000      8.000000        1.000000
5    recommendationservice-2  mesh.source.recommendationservice.jaeger-colle...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:57:00+08:00                 3             3      597.000000         0.00000   1194.000000    760.000000        1.000000
6    recommendationservice-1  mesh.source.recommendationservice.jaeger-colle...        high  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2        0.003550         0.00355      0.002100      0.000000        0.690476
7    recommendationservice-2  container.node-6.container_network_transmit_pa...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:57:00+08:00                 3             3      217.500000       182.00000    335.500000    246.500000        0.261663
8    recommendationservice-2  mesh.destination.frontend.recommendationservic...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2       18.000000        14.50000     23.500000     18.500000        0.216216
9    recommendationservice-2  mesh.destination.frontend.recommendationservic...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2       18.000000        14.50000     23.500000     18.500000        0.216216
10   recommendationservice-2  mesh.destination.frontend.recommendationservic...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2       18.000000        14.50000     23.500000     18.500000        0.216216
11   recommendationservice-2  mesh.source.recommendationservice.productcatal...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2       18.000000        14.50000     23.500000     18.500000        0.216216
12   recommendationservice-2  mesh.source.recommendationservice.productcatal...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2       18.000000        14.50000     23.500000     18.500000        0.216216
13   recommendationservice-2  mesh.source.recommendationservice.productcatal...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2       18.000000        14.50000     23.500000     18.500000        0.216216
14   recommendationservice-1  mesh.destination.frontend.recommendationservic...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2        0.059900         0.05175      0.085150      0.060375        0.142857
15  recommendationservice2-0  mesh.destination.frontend2.recommendationservi...        high  2022-03-21T04:34:00+08:00  2022-03-21T04:35:00+08:00                 2             2        0.124275         0.11720      0.111650      0.074125        0.113077
16  recommendationservice2-0  mesh.destination.frontend2.recommendationservi...        high  2022-03-21T04:34:00+08:00  2022-03-21T04:35:00+08:00                 2             2       35.500000        33.50000     32.500000     23.000000        0.092308
17  recommendationservice2-0  mesh.destination.frontend2.recommendationservi...        high  2022-03-21T04:34:00+08:00  2022-03-21T04:35:00+08:00                 2             2       35.500000        33.50000     32.500000     23.000000        0.092308
18  recommendationservice2-0  mesh.destination.frontend2.recommendationservi...        high  2022-03-21T04:34:00+08:00  2022-03-21T04:35:00+08:00                 2             2       35.500000        33.50000     32.500000     23.000000        0.092308
19  recommendationservice2-0  mesh.source.recommendationservice2.productcata...        high  2022-03-21T04:34:00+08:00  2022-03-21T04:35:00+08:00                 2             2       35.500000        33.50000     32.500000     23.000000        0.092308```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse existing variables if present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure ts column exists and is timezone-aware in Asia/Shanghai
if 'ts' not in metric_df.columns:
    if metric_df['timestamp'].dtype.kind in {'i', 'u', 'f'}:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], utc=True).dt.tz_convert(tz)
else:
    metric_df['ts'] = pd.to_datetime(metric_df['ts'], utc=True).dt.tz_convert(tz)

# Ensure numeric value column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Target filtering
target_cmdb = 'recommendationservice-2'
kpi_mask = metric_df['kpi_name'].str.contains('container_last_seen|container_fs_usage', na=False)
cmdb_mask = metric_df['cmdb_id'] == target_cmdb

# Incident window (inclusive)
start = tz.localize(pd.to_datetime("2022-03-21 04:30:00"))
end = tz.localize(pd.to_datetime("2022-03-21 05:00:00"))

window_mask = (metric_df['ts'] >= start) & (metric_df['ts'] <= end)

# Select rows
recsvc2_fs_lastseen_window = metric_df[cmdb_mask & kpi_mask & window_mask].copy()

# Add ISO timestamp column for readability
recsvc2_fs_lastseen_window['ts_iso'] = recsvc2_fs_lastseen_window['ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)

# Select requested columns and sort ascending by timestamp, limit to 20 rows
recsvc2_fs_lastseen_window = recsvc2_fs_lastseen_window[
    ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'ts_iso']
].sort_values('ts_iso', ascending=True).head(20).reset_index(drop=True)

recsvc2_fs_lastseen_window

```
Out[9]:
```
```python
# Summarize the extracted rows for recommendationservice-2 (container_fs_usage and container_last_seen)
df = recsvc2_fs_lastseen_window.copy()

# Separate KPIs
fs_mask = df['kpi_name'].str.contains('container_fs_usage', na=False)
ls_mask = df['kpi_name'].str.contains('container_last_seen', na=False)
fs_df = df[fs_mask]
ls_df = df[ls_mask]

# Compute simple stats
fs_min = float(fs_df['value'].min()) if not fs_df.empty else None
fs_max = float(fs_df['value'].max()) if not fs_df.empty else None
fs_delta = (fs_max - fs_min) if (fs_min is not None and fs_max is not None) else None
fs_count = fs_df.shape[0]

ls_unique = sorted(ls_df['value'].unique()) if not ls_df.empty else []
ls_count = ls_df.shape[0]

earliest_ts = df['ts_iso'].min() if not df.empty else None
latest_ts = df['ts_iso'].max() if not df.empty else None

summary = (
    "Summary for recommendationservice-2 (container_fs_usage & container_last_seen) in 2022-03-21 04:30–05:00 +08:\n"
    f"- Rows returned: {len(df)} (showing up to 20). Time range in output: {earliest_ts} to {latest_ts}.\n"
    f"- Filesystem usage (kpi contains 'container_fs_usage'): {fs_count} points; min={fs_min}, max={fs_max}, delta={fs_delta:.6f}.\n"
    f"  (Interpretation: FS usage slowly increases ~{fs_delta:.6f} MB over the sampled interval — no abrupt jump.)\n"
    f"- container_last_seen values ({ls_count} points) are stable/consistent: unique timestamps = {ls_unique}.\n"
    "- Conclusion: In this incident window the recommendationservice-2 filesystem usage shows a small steady rise and\n"
    "  container_last_seen is stable; there are no obvious sudden filesystem or heartbeat anomalies in these rows."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                  cmdb_id                                          kpi_name         value                     ts_iso
0   1647808200  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.129688e+01  2022-03-21T04:30:00+08:00
1   1647808200  recommendationservice-2              container.node-6.container_last_seen  1.647808e+09  2022-03-21T04:30:00+08:00
2   1647808260  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.130664e+01  2022-03-21T04:31:00+08:00
3   1647808260  recommendationservice-2              container.node-6.container_last_seen  1.647808e+09  2022-03-21T04:31:00+08:00
4   1647808320  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.131250e+01  2022-03-21T04:32:00+08:00
5   1647808320  recommendationservice-2              container.node-6.container_last_seen  1.647808e+09  2022-03-21T04:32:00+08:00
6   1647808380  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.131836e+01  2022-03-21T04:33:00+08:00
7   1647808380  recommendationservice-2              container.node-6.container_last_seen  1.647808e+09  2022-03-21T04:33:00+08:00
8   1647808440  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.132422e+01  2022-03-21T04:34:00+08:00
9   1647808440  recommendationservice-2              container.node-6.container_last_seen  1.647808e+09  2022-03-21T04:34:00+08:00
10  1647808500  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.132812e+01  2022-03-21T04:35:00+08:00
11  1647808500  recommendationservice-2              container.node-6.container_last_seen  1.647809e+09  2022-03-21T04:35:00+08:00
12  1647808560  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.133789e+01  2022-03-21T04:36:00+08:00
13  1647808560  recommendationservice-2              container.node-6.container_last_seen  1.647809e+09  2022-03-21T04:36:00+08:00
14  1647808620  recommendationservice-2              container.node-6.container_last_seen  1.647809e+09  2022-03-21T04:37:00+08:00
15  1647808620  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.134375e+01  2022-03-21T04:37:00+08:00
16  1647808680  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.135156e+01  2022-03-21T04:38:00+08:00
17  1647808680  recommendationservice-2              container.node-6.container_last_seen  1.647809e+09  2022-03-21T04:38:00+08:00
18  1647808740  recommendationservice-2  container.node-6.container_fs_usage_MB./dev/vda1  4.135742e+01  2022-03-21T04:39:00+08:00
19  1647808740  recommendationservice-2              container.node-6.container_last_seen  1.647809e+09  2022-03-21T04:39:00+08:00```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel tables
try:
    recsvc_fault_segments
except NameError:
    raise RuntimeError("recsvc_fault_segments not found in kernel. Please run the recsvc segmentation step first.")
try:
    frontend2_fault_segments
except NameError:
    raise RuntimeError("frontend2_fault_segments not found in kernel. Please run the frontend-2 segmentation step first.")

# Prepare copies
rec_df = recsvc_fault_segments.copy()
f2_df = frontend2_fault_segments.copy()

# Ensure start_ts/end_ts are datetime for ordering; they may be ISO strings with tz info
for df in (rec_df, f2_df):
    if 'start_ts' in df.columns:
        df['start_dt'] = pd.to_datetime(df['start_ts'], utc=True).dt.tz_convert(tz)
    else:
        df['start_dt'] = pd.NaT
    if 'end_ts' in df.columns:
        df['end_dt'] = pd.to_datetime(df['end_ts'], utc=True).dt.tz_convert(tz)
    else:
        df['end_dt'] = pd.NaT

# Extract earliest segment for recommendationservice-2
rec_earliest = rec_df[rec_df['cmdb_id'] == 'recommendationservice-2'].copy()
if not rec_earliest.empty:
    rec_earliest = rec_earliest.sort_values('start_dt').head(1)
    rec_earliest['cmdb_id'] = 'recommendationservice-2'
else:
    rec_earliest = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','breach_type','start_ts','end_ts','duration_minutes','count_points','severity_ratio'
    ])

# Extract earliest segment for frontend-2
# frontend2_fault_segments may not contain a cmdb_id column; add it if missing
if 'cmdb_id' not in f2_df.columns:
    f2_df['cmdb_id'] = 'frontend-2'
f2_earliest = f2_df[f2_df['cmdb_id'] == 'frontend-2'].copy()
if not f2_earliest.empty:
    f2_earliest = f2_earliest.sort_values('start_dt').head(1)
else:
    f2_earliest = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','breach_type','start_ts','end_ts','duration_minutes','count_points','severity_ratio'
    ])

# Select and normalize columns for output
select_cols = ['cmdb_id','kpi_name','breach_type','start_ts','end_ts','duration_minutes','count_points','severity_ratio']
out_rec = rec_earliest[select_cols].copy() if not rec_earliest.empty else rec_earliest
out_f2 = f2_earliest[select_cols].copy() if not f2_earliest.empty else f2_earliest

# Combine the two rows (up to 2)
earliest_segments = pd.concat([out_rec, out_f2], ignore_index=True)[select_cols]

# Final result variable for display
earliest_segments

```
Out[10]:
```
```python
# Compose a concise plain-English summary of the earliest anomalous segments found for recommendationservice-2 and frontend-2.
try:
    earliest_segments
except NameError:
    raise RuntimeError("Expected variable 'earliest_segments' not found in kernel.")

rows = earliest_segments.copy()
if rows.empty:
    summary_text = "No multi-point anomalous segments found for recommendationservice-2 or frontend-2 in the incident window."
else:
    lines = ["Earliest anomalous segments (one per component):"]
    for _, r in rows.iterrows():
        lines.append(
            f"- {r['cmdb_id']}: KPI='{r['kpi_name']}', breach_type={r['breach_type']}, "
            f"start={r['start_ts']}, end={r['end_ts']}, duration_minutes={int(r['duration_minutes'])}, "
            f"points={int(r['count_points'])}, severity_ratio={float(r['severity_ratio']):.3f}"
        )
    summary_text = "\n".join(lines)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name breach_type                   start_ts                     end_ts  duration_minutes  count_points  severity_ratio
0  recommendationservice-2  mesh.destination.frontend.recommendationservic...         low  2022-03-21T04:55:00+08:00  2022-03-21T04:56:00+08:00                 2             2        0.216216
1               frontend-2  container.node-5.container_network_receive_MB....        high  2022-03-21T04:41:00+08:00  2022-03-21T04:42:00+08:00                 2             2        2.331847```
```